In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *
from libs.prism_diagnostics_helpers import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [8]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_prism, prism.root_prism.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



(PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism'), True)

### Running prism

In [9]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

Loaded /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism/deconv.h5ad (8.1 MB)
1918


In [10]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [11]:
res.theta_type.columns

Index(['Acinar cell', 'B cell', 'Ductal cell type 1', 'Endocrine cell', 'Endothelial cell',
       'Fibroblast cell', 'Macrophage cell', 'Stellate cell', 'T cell', 'malignant'],
      dtype='object')

In [12]:
print(res.theta.shape)
res.theta.head(5)

(153, 10)


,Fibroblast cell,Stellate cell,Macrophage cell,Endothelial cell,T cell,B cell,Ductal cell type 2,Endocrine cell,Ductal cell type 1,Acinar cell
T-C3L-02890,0.603,8.666e-03,0.025,0.029,8.828e-73,0.018,0.105,1.176e-152,1.083e-01,0.104
T-C3L-03635,0.760,5.456e-03,0.016,0.028,2.978e-163,0.008,0.183,4.146e-105,2.173e-72,0.000
T-C3L-02701,0.763,1.275e-08,0.027,0.015,1.957e-222,0.005,0.175,1.414e-02,1.685e-148,0.000
T-C3L-04072,0.296,1.671e-02,0.076,0.034,1.428e-144,0.036,0.514,0.000e+00,2.690e-02,0.001
T-C3L-00589,0.384,4.344e-02,0.048,0.036,2.727e-125,0.014,0.422,1.177e-296,1.481e-02,0.038


In [13]:
res.theta.tail(5)

,Fibroblast cell,Stellate cell,Macrophage cell,Endothelial cell,T cell,B cell,Ductal cell type 2,Endocrine cell,Ductal cell type 1,Acinar cell
N-C3L-02606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N-C3N-03173,0.003,0.040,0.002,0.007,2.048e-85,4.374e-03,0.929,0.000,0.013,3.553e-114
N-C3N-02696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N-TCGA-H6-8124,0.581,0.014,0.089,0.030,4.796e-138,4.272e-16,0.094,0.004,0.153,3.388e-02
N-TCGA-H6-A45N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [14]:
res.cell_type_expression("Ductal cell type 1").shape

(1918, 153)

In [15]:
res.cell_type_expression("Ductal cell type 1").head(3)

,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
ENSG00000000938,51.301,NaN,NaN,37.533,71.738,133.775,NaN,103.585,65.544,NaN,...,NaN,42.916,38.229,NaN,NaN,NaN,39.535,NaN,81.539,NaN
ENSG00000001617,46.895,NaN,NaN,139.224,37.504,111.861,NaN,48.340,100.671,NaN,...,NaN,56.004,34.789,NaN,NaN,NaN,33.189,NaN,84.933,NaN
ENSG00000001626,98976.750,NaN,NaN,60517.504,64058.141,12317.005,NaN,85658.117,93382.445,NaN,...,NaN,44093.203,37337.992,NaN,NaN,NaN,32737.848,NaN,77932.102,NaN


In [16]:
res.cell_type_expression("Ductal cell type 2").shape

(1918, 153)

In [17]:
res.cell_type_expression("Ductal cell type 2").head(3)

,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
ENSG00000000938,105.698,49.471,169.428,75.978,130.365,249.999,127.351,187.454,110.75,131.444,...,NaN,60.164,65.509,NaN,NaN,NaN,95.958,NaN,204.643,NaN
ENSG00000001617,211.804,188.735,283.416,617.821,149.403,458.264,447.011,191.767,372.90,247.584,...,NaN,172.114,130.685,NaN,NaN,NaN,176.593,NaN,467.284,NaN
ENSG00000001626,4642.306,1529.108,865.661,2788.810,2650.019,523.997,1481.312,3528.819,3592.04,1061.946,...,NaN,1407.204,1456.537,NaN,NaN,NaN,1808.896,NaN,4452.575,NaN


### 2. theta is now fixed -> expand Z to every gene

In [18]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

verbose=True
force=False

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        gene_key="geneid", force=force, verbose=verbose)
force=False
verbose=True
fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

filename_ad = prism.root_prism / fname_ad
compression = "gzip"

verbose=True
fname_celltype = "all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file
Table opened ((27177, 153)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism/bulk_matrix_geneid.tsv'
Table opened ((153, 4)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism/bulk_metadata_geneid.tsv'
57,530 cells x 24,005 genes | obs: []
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 2447
Acinar cell            1935
Endocrine cell          729
Name: count, dtype

### Bulk - by geneid

In [19]:
#--- reference geneid --> 
gene_map = prism.load_gene_map("geneid")
print(df_bulk.shape)
df_bulk.head(2)

(27177, 153)


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
geneid,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,1486,2083,1558,546,1208,648,896,1532,821,1217,...,891,1063,1261,1821,554,1244,977,1576,3738,369
ENSG00000000005,12,97,15,1,14,2,5,6,3,11,...,0,7,2,6,1,1,3,39,4,5


In [20]:
bulk_symbs = pd.read_csv(prism.root_prism / "bulk_matrix.tsv", sep="\t", index_col=0, usecols=[0])

### ref_new --> new reference, by geneid (ensbeml)

In [21]:
force=False
verbose=False

ref_new, df_to_from = prism.harmonize_reference_to_ensembl(bulk_symbs=bulk_symbs, ref=ref, gene_map=gene_map, force=False, verbose=verbose)

print(ref_new.shape)
print(df_to_from.status.value_counts())
ref_new.head(2)

(10, 17540)
status
ok               17540
no_ensembl_id     6249
Name: count, dtype: int64


,ENSG00000225880,ENSG00000187634,ENSG00000188976,ENSG00000187961,ENSG00000187583,ENSG00000188290,ENSG00000187608,ENSG00000188157,ENSG00000237330,ENSG00000131591,...,ENSG00000167355,ENSG00000184999,ENSG00000235910,ENSG00000157335,ENSG00000257008,ENSG00000166573,ENSG00000268182,ENSG00000254453,ENSG00000226245,ENSG00000213424
cell_state,,,,,,,,,,,,,,,,,,,,,
Fibroblast cell,214.0,1246.0,2337.0,115.0,90.0,4981.0,18665.0,1758.0,11.0,274.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Stellate cell,162.0,112.0,1719.0,52.0,26.0,9730.0,16767.0,1611.0,2.0,172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


### ref_new, s2t

In [22]:
print(ref_new.shape)
ref_new.head(3)

(10, 17540)


,ENSG00000225880,ENSG00000187634,ENSG00000188976,ENSG00000187961,ENSG00000187583,ENSG00000188290,ENSG00000187608,ENSG00000188157,ENSG00000237330,ENSG00000131591,...,ENSG00000167355,ENSG00000184999,ENSG00000235910,ENSG00000157335,ENSG00000257008,ENSG00000166573,ENSG00000268182,ENSG00000254453,ENSG00000226245,ENSG00000213424
cell_state,,,,,,,,,,,,,,,,,,,,,
Fibroblast cell,214.0,1246.0,2337.0,115.0,90.0,4981.0,18665.0,1758.0,11.0,274.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Stellate cell,162.0,112.0,1719.0,52.0,26.0,9730.0,16767.0,1611.0,2.0,172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
Macrophage cell,182.0,35.0,1293.0,66.0,47.0,918.0,19924.0,825.0,9.0,157.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
print(type(s2t), len(s2t))
s2t

<class 'pandas.core.series.Series'> 10


cell_state
Fibroblast cell          Fibroblast cell
Stellate cell              Stellate cell
Macrophage cell          Macrophage cell
Endothelial cell        Endothelial cell
T cell                            T cell
B cell                            B cell
Ductal cell type 2             malignant
Endocrine cell            Endocrine cell
Ductal cell type 1    Ductal cell type 1
Acinar cell                  Acinar cell
Name: cell_type, dtype: object

### why Zfull resulted in 16550 genes?

Because full_Z reconstructs the full gene set, not the subset BayesPrism fitted on.

The three numbers you've seen trace it:

- 16550 — genes in full_Z, the whole expression matrix
- 1604 — genes in cell_type_expression, the marker-based fit
- ~10000 — after min_share/min_counts filtering

BayesPrism runs on gene_subset (marker/signature genes) for tractability and identifiability. That gave 1604. 

full_Z then projects the remaining ~15000 genes onto the fitted compartment basis — which is precisely why you built it: to recover the lncRNA/antisense loci (FAM83A-AS1, HOXA10-AS, HOXB-AS3/4, MIR7-3HG) that the marker fit excluded.

Shape is (153 samples, 10 cell types, 16550 genes) — build_ms_from_full_Z resolves that axis order automatically.

The consequence you should hold onto: 
- those ~15000 recovered genes are not Gibbs posterior estimates. 
- they're projections onto a basis fitted from 1604 genes, 
- so their sampling variance is structurally different 
  - no posterior shrinkage in the same sense, 
  - and their between-sample variation partly reflects the projection rather than compartment-specific evidence.

In [24]:
Zfull, genes_full = prism.full_Z(res, df_bulk, ref_new)
print(Zfull.shape)

Zfull[0][3][:5]

(153, 10, 17540)


array([54.085945,  6.94367 , 42.73008 , 46.775616, 13.698739],
      dtype=float32)

In [25]:
dic = {}

for cell_state in res.states:
    Z = prism.state_expression(Zfull, genes_full, res, cell_state)
    dic[cell_state] = Z
    print(cell_state, Z.shape)


Fibroblast cell (17540, 153)
Stellate cell (17540, 153)
Macrophage cell (17540, 153)
Endothelial cell (17540, 153)
T cell (17540, 153)
B cell (17540, 153)
Ductal cell type 2 (17540, 153)
Endocrine cell (17540, 153)
Ductal cell type 1 (17540, 153)
Acinar cell (17540, 153)


In [26]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

Fibroblast cell


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
ENSG00000000003,3.207e+01,5.801e+01,4.361e+01,32.366,3.406e+01,31.881,3.565e+01,5.433e+01,30.481,40.203,...,NaN,16.647,16.364,NaN,NaN,NaN,27.003,NaN,53.037,NaN
ENSG00000000005,1.556e-01,1.310e+00,2.711e-01,0.029,1.720e-01,0.058,1.358e-01,8.509e-02,0.060,0.182,...,NaN,0.124,0.020,NaN,NaN,NaN,0.119,NaN,0.035,NaN
ENSG00000000419,4.123e+01,4.310e+01,4.238e+01,62.828,4.240e+01,43.955,2.572e+01,4.332e+01,42.273,46.569,...,NaN,25.995,36.263,NaN,NaN,NaN,21.046,NaN,32.622,NaN
ENSG00000000457,3.290e+01,5.382e+01,2.546e+01,31.656,3.979e+01,32.726,2.998e+01,3.380e+01,31.080,40.644,...,NaN,16.563,21.391,NaN,NaN,NaN,17.661,NaN,15.153,NaN
ENSG00000000460,1.502e+01,2.333e+01,8.703e+00,16.155,1.337e+01,12.151,8.187e+00,9.839e+00,12.122,12.879,...,NaN,7.853,9.335,NaN,NaN,NaN,6.459,NaN,4.055,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000287971,2.388e-09,6.709e-10,6.899e-10,0.000,9.010e-10,0.000,8.077e-10,3.939e-09,0.000,0.000,...,NaN,0.000,0.000,NaN,NaN,NaN,0.000,NaN,0.000,NaN
ENSG00000288302,8.825e-02,7.354e-02,1.746e-01,0.000,4.293e-01,0.094,0.000e+00,7.508e-02,0.054,0.516,...,NaN,0.000,0.364,NaN,NaN,NaN,5.905,NaN,0.191,NaN
ENSG00000288547,6.591e-01,3.459e-01,3.643e-01,0.615,4.794e-01,0.212,4.817e-01,3.034e-01,0.131,0.509,...,NaN,0.914,0.099,NaN,NaN,NaN,0.724,NaN,4.869,NaN
ENSG00000288596,1.110e+01,1.045e+01,5.792e+00,9.570,1.071e+01,8.293,8.219e+00,8.764e+00,8.785,8.110,...,NaN,9.559,5.357,NaN,NaN,NaN,6.290,NaN,2.457,NaN


### Ductal 2 - malignant

In [27]:
Zmal = prism.state_expression(Zfull, genes_full, res, "Ductal cell type 2")

In [28]:
gene_map.head(2)

,symbol,biotype
geneid,,
ENSG00000000003,TSPAN6,protein_coding
ENSG00000000005,TNMD,protein_coding


In [29]:
gene_map = prism.load_gene_map("geneid")
gene_map_symb = gene_map.reset_index().copy()
gene_map_symb.set_index("symbol", inplace=True)
gene_map_symb.head(2)

,geneid,biotype
symbol,,
TSPAN6,ENSG00000000003,protein_coding
TNMD,ENSG00000000005,protein_coding


In [30]:
for g in [gene_map_symb.loc[x].geneid for x in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"] ]:
    if g in genes_full:
        print(g, prism.gene_compartment_share(Zfull, genes_full, res, g).head(3).round(3).to_dict())

ENSG00000204949 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000253187 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000233101 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000176840 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}


In [31]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]

prog2 = ["GATA6", "KRT17",  "H19", "DLEU1", "DLEU2"]

In [32]:
prog1_ens = []

for g in prog1:
    try:
        prog1_ens.append(gene_map_symb.loc[g].geneid)
    except KeyError:
        print(f"Could not find {g}")

prog1_ens

['ENSG00000204949',
 'ENSG00000253187',
 'ENSG00000233101',
 'ENSG00000242207',
 'ENSG00000176840']

In [33]:
prog2_ens = []

for g in prog2:
    try:
        prog2_ens.append(gene_map_symb.loc[g].geneid)
    except KeyError:
        print(f"Could not find {g}")

prog2_ens

['ENSG00000141448',
 'ENSG00000128422',
 'ENSG00000130600',
 'ENSG00000176124',
 'ENSG00000231607']

In [34]:
df_bulk.head(2)

,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
geneid,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,1486,2083,1558,546,1208,648,896,1532,821,1217,...,891,1063,1261,1821,554,1244,977,1576,3738,369
ENSG00000000005,12,97,15,1,14,2,5,6,3,11,...,0,7,2,6,1,1,3,39,4,5


In [35]:
Zfull.shape

(153, 10, 17540)

In [36]:
len(genes_full)

17540

In [37]:
genes_full[:5]

['ENSG00000000003',
 'ENSG00000000005',
 'ENSG00000000419',
 'ENSG00000000457',
 'ENSG00000000460']

In [38]:
for g in prog1_ens:
    if g in genes_full:
        print(g, prism.gene_compartment_share(Zfull, genes_full, res, g).head(3).round(3).to_dict())

ENSG00000204949 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000253187 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000233101 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000176840 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}


In [39]:
for g in prog2_ens:
    if g in genes_full:
        print(g, prism.gene_compartment_share(Zfull, genes_full, res, g).head(3).round(3).to_dict())

ENSG00000141448 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000128422 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
ENSG00000176124 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}


### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [40]:
[g for g in prog1 if g in df_bulk.index]

[]

### present in the scRNA reference?

In [41]:
  
[g for g in prog1_ens if g in ref_new.columns]

['ENSG00000204949', 'ENSG00000253187', 'ENSG00000233101', 'ENSG00000176840']

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [42]:
adata.obs

,cluster,cell_type,cell_state
cell,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell
...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1


In [43]:
import re, numpy as np, pandas as pd

adata.obs["sample"] = adata.obs_names.to_series().str.extract(r"^([TN]\d+)_")[0].values
adata.obs["tissue"] = np.where(adata.obs["sample"].str.startswith("T"), "tumor", "normal")

print("")
print(adata.obs.groupby("tissue")["sample"].nunique())      # expect tumor 24, normal 11
print("")
print(pd.crosstab(adata.obs["cell_state"], adata.obs["tissue"]))


tissue
normal    11
tumor     24
Name: sample, dtype: int64

tissue              normal  tumor
cell_state                       
Acinar cell           1423    512
B cell                  31   2416
Ductal cell type 1    7671   2646
Ductal cell type 2       0  11315
Endocrine cell         270    459
Endothelial cell      3983   5134
Fibroblast cell        940   5802
Macrophage cell        559   4802
Stellate cell          623   5284
T cell                  44   3616


### Count Malignant Cells - accordingo to transcriptomics

In [44]:
d2 = adata.obs["cell_state"].eq("Ductal cell type 2")
print(len(d2))
d2[:5]

57530


cell
T1_AAACCTGAGATGTCGG    False
T1_AAACGGGGTCATGCAT    False
T1_AAAGATGCATGTTGAC    False
T1_AAAGATGGTCGAGTTT    False
T1_AAAGATGGTCTCTCTG    False
Name: cell_state, dtype: bool

In [45]:
is_t = adata.obs["tissue"].eq("tumor")
print(np.sum(is_t))

41986


In [46]:
adata.obs["cell_state"] = np.where(d2 &  is_t, "Malignant ductal",
                          np.where(d2 & ~is_t, "Ductal cell type 2 normal",
                                   adata.obs["cell_state"]))
adata.obs

,cluster,cell_type,cell_state,sample,tissue
cell,,,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell,T1,tumor
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell,T1,tumor
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell,T1,tumor
...,...,...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell,N11,normal
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell,N11,normal
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1,N11,normal


In [47]:
from collections import Counter

Counter(adata.obs["cell_state"] )

Counter({'Malignant ductal': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [48]:
adata.obs["cell_type"]  = np.where(adata.obs["cell_state"].eq("Malignant ductal"),
                                   "malignant", adata.obs["cell_type"])

Counter(adata.obs["cell_type"] )

Counter({'malignant': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [49]:
ref2, s2t = prism.pseudobulk_reference(adata, state_key="cell_state", type_key="cell_type")
s2t.to_dict()

{'Fibroblast cell': 'Fibroblast cell',
 'Stellate cell': 'Stellate cell',
 'Macrophage cell': 'Macrophage cell',
 'Endothelial cell': 'Endothelial cell',
 'T cell': 'T cell',
 'B cell': 'B cell',
 'Malignant ductal': 'malignant',
 'Endocrine cell': 'Endocrine cell',
 'Ductal cell type 1': 'Ductal cell type 1',
 'Acinar cell': 'Acinar cell'}

### LFC

calc_celltype_lfc() — each compartment vs the mean of the others, paired across samples by default. Paired is the right default here because every sample contributes every cell type, so pairing removes cohort/purity variance. This doubles as deconvolution QC: if the ductal compartment doesn't recover KRT19/TFF1/CEACAM6 and fibroblast doesn't recover COL1A1/POSTN, θ or the Peng reference is off and step 2 is meaningless.

### Critics

- Why not, for each cell type, tumor samples x normal samples
- Only Ductal 2 Tumor has no normal samples - to confirm


In [50]:
res.__dict__.keys()

dict_keys(['theta', 'theta_stage1', 'theta_type', 'tumor_purity', 'genes', 'Z', 'states'])

In [51]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

### MalignantCluster

In [52]:
import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

0.35.0


In [53]:
root_mprog_disease = cbio.root_mprog_disease
mal_cell_name = "Ductal cell type 2"

mc = pml.MalignantCluster(prism=prism, res=res, 
                          df_bulk=df_bulk, ref=ref_new, 
                          root_mprog_disease = root_mprog_disease,
                          mal_cell_name = mal_cell_name,
                          organ="Pancreas")

mc

### Turn compartment counts into a matrix it is safe to cluster.

In [54]:
X, diag = mc.prepare_malignant_matrix(decouple_purity=False, keep_genes=mc.program1_panel, drop_pattern=r"^N-")
print(X.shape)
X.head(3)

excluded 22/153 samples by keep_samples/drop_pattern
(118, 2000)


,ENSG00000001084,ENSG00000001561,ENSG00000002587,ENSG00000002726,ENSG00000002834,ENSG00000003147,ENSG00000003249,ENSG00000003400,ENSG00000004478,ENSG00000005001,...,ENSG00000274211,ENSG00000275342,ENSG00000275395,ENSG00000275410,ENSG00000276180,ENSG00000277363,ENSG00000277972,ENSG00000278259,ENSG00000278535,ENSG00000278540
T-C3L-02890,6.780,5.736,7.171,5.541,8.460,7.229,3.217,7.004,6.726,7.630,...,6.132,6.505,8.483,5.027,4.268,2.718,4.797,7.108,3.834,7.867
T-C3L-03635,6.269,5.560,7.215,6.130,8.503,8.109,4.257,6.930,7.244,7.363,...,6.472,5.909,6.709,6.371,3.522,3.931,4.802,7.070,4.182,8.450
T-C3L-02701,5.989,7.098,7.601,7.242,8.451,7.274,4.592,7.107,7.048,6.365,...,6.274,5.258,7.830,4.836,4.575,3.923,5.229,7.101,4.260,7.952


In [55]:
X.tail(3)

,ENSG00000001084,ENSG00000001561,ENSG00000002587,ENSG00000002726,ENSG00000002834,ENSG00000003147,ENSG00000003249,ENSG00000003400,ENSG00000004478,ENSG00000005001,...,ENSG00000274211,ENSG00000275342,ENSG00000275395,ENSG00000275410,ENSG00000276180,ENSG00000277363,ENSG00000277972,ENSG00000278259,ENSG00000278535,ENSG00000278540
T-TCGA-3A-A9IV,7.565,6.023,3.324,1.448,8.007,7.386,2.533,2.697,6.865,0.420,...,7.164,3.547,6.159,0.722,2.338,6.233,6.055,6.180,5.108,7.149
T-TCGA-2J-AABT,6.735,5.751,5.384,9.160,8.409,7.006,5.090,6.294,6.789,7.343,...,5.424,6.412,6.549,6.501,2.859,4.072,6.318,5.870,5.375,7.263
T-TCGA-H6-A45N,5.685,5.722,7.437,7.615,8.812,6.507,5.103,6.228,6.752,7.519,...,5.128,6.746,7.716,6.657,3.044,4.665,6.183,5.564,4.241,6.559


In [56]:
lista = [x for x in X.index if x.startswith('T-')]
X.shape[0], len(lista) == X.shape[0]

(118, True)

In [57]:
diag.keys()

dict_keys(['samples_excluded_by_filter', 'samples_dropped', 'n_genes_expressed', 'n_genes_share_not_computable', 'n_genes_share_ok', 'forced_genes_status', 'n_genes_kept', 'n_hvg', 'pc_theta_pearson_raw', 'pc_theta_pearson', 'decouple_purity', 'pc_theta_note', 'sample_mean_expr', 'sample_total_Z', 'theta_mal', 'n_samples_used', 'theta_excluded', 'theta_kept'])

In [58]:
diag["samples_excluded_by_filter"]

['N-C3L-04072',
 'N-C3L-00589',
 'N-C3L-03123',
 'N-C3L-04080',
 'N-C3L-00640',
 'N-C3N-01719',
 'N-C3L-07033',
 'N-C3L-00819',
 'N-C3L-07032',
 'N-C3L-01689',
 'N-C3N-01899',
 'N-C3N-00517',
 'N-C3N-03069',
 'N-C3N-02765',
 'N-C3L-07037',
 'N-C3N-02589',
 'N-C3N-02996',
 'N-C3L-02606',
 'N-C3N-03173',
 'N-C3N-02696',
 'N-TCGA-H6-8124',
 'N-TCGA-H6-A45N']

### pc_theta_pearson and pc_theta_pearson_raw

**The computation.** Run PCA on the samples × genes matrix, take the first 5 principal components, and correlate each PC's sample scores with `theta_mal` (each sample's malignant fraction). `pcs[:, i]` is one number per sample for PC *i*; `theta_mal.values` is one number per sample. `np.corrcoef(...)[0,1]` pulls the off-diagonal — the Pearson r between them.

You get 5 numbers, one per PC. Each answers: *does this dominant axis of variation track tumour purity?*

**The two versions:**

| | matrix | meaning |
|---|---|---|
| `pc_theta_pearson_raw` | `logx` — log2-CPM before decoupling | how much purity is in the data |
| `pc_theta_pearson` | `Xc` — the matrix you actually cluster | how much purity survives into the analysis |

With `decouple_purity=False` they're the same matrix, so the numbers match — your `[-0.596, -0.205, 0.227, -0.359, -0.158]` versus `[-0.596, -0.205, 0.228, -0.359, -0.159]`. The tiny differences are HVG selection, which happens between the two calls.

With `decouple_purity=True`, `Xc` holds residuals from regressing on `theta_mal`, and residuals are **orthogonal to their regressors by construction**. So `pc_theta_pearson` becomes ~1e-15 — pure floating-point noise. It proves the arithmetic worked, nothing about your data. That's why 0.20.1 added the `_raw` version and the `pc_theta_note`: I originally had you reading a number that can only ever be zero.

**Your actual numbers matter.** PC1 at r = −0.596 means ~36% of the leading component's variance is shared with purity, and PC4 at −0.359 adds more. The sign says low-purity samples score high on PC1. Since `X` is what produced the consensus clustering, the 134-gene tumour axis, and the 6/119 splits, purity is a live confound in all of them.

Which is the concrete reason to run `decouple_purity=True` and compare — with the standing caveat that basal-like PDAC is genuinely lower-purity, so some of that r is biology you'd be deleting.

In [59]:
diag["pc_theta_pearson"]  # PC-vs-theta_mal

[-0.6396089330262075,
 -0.22766415753793454,
 0.234662042184136,
 -0.31802866389422885,
 -0.13013308386250083]

In [60]:
diag["pc_theta_pearson_raw"]   # PC-vs-theta_mal on logx (pre-decoupling)

[-0.6396006275175102,
 -0.22759702118189976,
 0.2347338478839774,
 -0.3180944642716501,
 -0.1305640943826249]

In [61]:
diag["pc_theta_note"]          # warns the decoupled version is ~0 by construction

'decouple_purity=False, so pc_theta_pearson and pc_theta_pearson_raw are the same matrix and both are informative: a large |r| on an early PC means the clustering is tracking tumour purity.'

In [62]:
diag["sample_mean_expr"]       # Xc.mean(axis=1) per sample

T-C3L-02890       6.171
T-C3L-03635       6.080
T-C3L-02701       6.174
T-C3L-04072       5.682
T-C3L-00589       6.050
                  ...  
T-TCGA-2L-AAQM    4.425
T-TCGA-3A-A9IR    4.159
T-TCGA-3A-A9IV    4.590
T-TCGA-2J-AABT    6.073
T-TCGA-H6-A45N    6.231
Length: 118, dtype: float32

In [63]:
diag["sample_total_Z"]         # ms.Z.sum(axis=1) per sample

T-C3L-02890       1.000e+06
T-C3L-03635       1.000e+06
T-C3L-02701       1.000e+06
T-C3L-04072       1.000e+06
T-C3L-00589       1.000e+06
                    ...    
T-TCGA-2L-AAQM    1.000e+06
T-TCGA-3A-A9IR    1.000e+06
T-TCGA-3A-A9IV    1.000e+06
T-TCGA-2J-AABT    1.000e+06
T-TCGA-H6-A45N    1.000e+06
Length: 118, dtype: float32

In [64]:
diag["theta_excluded"]

count    15.000
mean      0.291
std       0.416
min       0.000
25%       0.002
50%       0.066
75%       0.563
max       0.985
Name: Ductal cell type 2, dtype: float64

In [65]:
diag["theta_kept"]

count    130.000
mean       0.345
std        0.256
min        0.000
25%        0.145
50%        0.298
75%        0.496
max        1.000
Name: Ductal cell type 2, dtype: float64

### Inspecting vars

In [66]:
import inspect
print(pml.__version__)
print("drop_pattern" in inspect.signature(mc.prepare_malignant_matrix).parameters)

0.35.0
True


In [67]:
info = mc.inspect_de_schema(genes=X.columns)
print(info.keys())
info["columns"]

dict_keys(['shard_file', 'file_mb', 'total_rows_in_shard', 'columns', 'dtypes', 'head', 'distinct_gene_name', 'n_distinct_gene_name', 'distinct_baseMean', 'n_distinct_baseMean', 'distinct_log2FoldChange', 'n_distinct_log2FoldChange', 'distinct_lfcSE', 'n_distinct_lfcSE', 'distinct_stat', 'n_distinct_stat', 'distinct_pvalue', 'n_distinct_pvalue', 'distinct_padj', 'n_distinct_padj', 'distinct_plate', 'n_distinct_plate', 'distinct_n_cells_trt', 'n_distinct_n_cells_trt', 'distinct_n_cells_ctrl', 'n_distinct_n_cells_ctrl', 'distinct_Cell_ID_Cellosaur', 'n_distinct_Cell_ID_Cellosaur', 'distinct_Cell_ID_DepMap', 'n_distinct_Cell_ID_DepMap', 'distinct_drug', 'n_distinct_drug', 'distinct_concentration', 'n_distinct_concentration', 'distinct_concentration_unit', 'n_distinct_concentration_unit', 'distinct_Cell_Name_Vevo', 'n_distinct_Cell_Name_Vevo', 'cell_line_metadata_columns', 'MATCH cell_line_metadata.Cell_ID_Cellosaur -> DE.Cell_ID_Cellosaur', 'MATCH cell_line_metadata.cell_name -> DE.Cell_N

['gene_name',
 'baseMean',
 'log2FoldChange',
 'lfcSE',
 'stat',
 'pvalue',
 'padj',
 'plate',
 'n_cells_trt',
 'n_cells_ctrl',
 'Cell_ID_Cellosaur',
 'Cell_ID_DepMap',
 'drug',
 'concentration',
 'concentration_unit',
 'Cell_Name_Vevo']

In [68]:
info["matches"]

['MATCH cell_line_metadata.Cell_ID_Cellosaur -> DE.Cell_ID_Cellosaur',
 'MATCH cell_line_metadata.cell_name -> DE.Cell_Name_Vevo']

In [69]:
info["dtypes"]

{'gene_name': 'object',
 'baseMean': 'float32',
 'log2FoldChange': 'float32',
 'lfcSE': 'float32',
 'stat': 'float32',
 'pvalue': 'float32',
 'padj': 'float32',
 'plate': 'object',
 'n_cells_trt': 'int64',
 'n_cells_ctrl': 'int64',
 'Cell_ID_Cellosaur': 'object',
 'Cell_ID_DepMap': 'object',
 'drug': 'object',
 'concentration': 'float32',
 'concentration_unit': 'object',
 'Cell_Name_Vevo': 'object'}

In [70]:
info["resolved_columns"]

{'gene': 'gene_name',
 'stat': 'stat',
 'cell_line': 'Cell_ID_Cellosaur',
 'drug': 'drug'}

In [71]:
info["numeric_profile"]      # min / max / mean / frac_negative / n_unique


,min,max,mean,frac_negative,n_unique
baseMean,0.000,95136.398,36.460,0.000,69136
log2FoldChange,-4.858,7.180,0.081,0.232,68458
lfcSE,0.007,4.425,1.248,0.000,68449
stat,-46.548,73.326,0.019,0.232,68488
pvalue,0.000,1.000,0.470,0.000,68433
padj,0.000,1.000,0.586,0.000,23253
n_cells_trt,1378.000,2165.000,1745.045,0.000,4
n_cells_ctrl,4862.000,4862.000,4862.000,0.000,1
concentration,0.050,0.050,0.050,0.000,1


In [72]:
info["signed_candidates"]

['log2FoldChange', 'stat']

In [73]:
cov = mc.index_coverage()
cov["n_lines_seen"], cov["n_lines_in_metadata"]

(50, 102)

In [74]:
cov["block_probe_counts"]        # min probes per block

count    50.0
mean      3.4
std       0.5
min       3.0
25%       3.0
50%       3.0
75%       4.0
max       4.0
Name: count, dtype: float64

In [75]:
lista = cov["in_metadata_not_in_index"]
len(lista), lista[:5]

(52, ['CVCL_0025', 'CVCL_0031', 'CVCL_0039', 'CVCL_0060', 'CVCL_0078'])

In [76]:
cov["n_lines_seen"], cov["n_lines_in_metadata"]

(50, 102)

In [77]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
    "acinar": "Acinar cell",
}

### Batch correction

In [84]:
import libs.prism_diagnostics_helpers as pdh
importlib.reload(pdh)

gene_len = pdh.calc_gene_lengths(cbio.root_colab)
type(gene_len), len(gene_len)

(collections.defaultdict, 78321)

In [87]:
gene_lengths = pd.Series(gene_len, name="exonic_len")
gene_lengths.iloc[:3]

ENSG00000290825    1762
ENSG00000223972     632
ENSG00000310526    2843
Name: exonic_len, dtype: int64

In [88]:
batch = pd.Series(np.where(mc.df_theta.index.str.contains("TCGA"), "TCGA", "CPTAC"),
                  index=mc.df_theta.index, name="cohort")
print(batch.value_counts().to_dict())

{'TCGA': 84, 'CPTAC': 69}


In [91]:
MIN_SHARE, MIN_COUNTS = 0.3, 10

# cohort label, indexed by sample — must cover every sample in df_theta
batch = pd.Series(np.where(mc.df_theta.index.str.contains("TCGA"), "TCGA", "CPTAC"),
                  index=mc.df_theta.index, name="cohort")
print(batch.value_counts().to_dict(), "\n")

X_fib2 = mc.compartment_matrix(cmap["fibroblast"], min_share=MIN_SHARE, min_counts=MIN_COUNTS, batch=batch)
X_mal2 = mc.compartment_matrix(cmap["malignant"],  min_share=MIN_SHARE, min_counts=MIN_COUNTS, batch=batch)
X_aci2 = mc.compartment_matrix(cmap["acinar"],     min_share=MIN_SHARE, min_counts=MIN_COUNTS, batch=batch)

for nm, Xc in [("fibroblast", X_fib2), ("malignant", X_mal2), ("acinar", X_aci2)]:
    print(f"{nm:<12} {Xc.shape[0]:>4} samples x {Xc.shape[1]:>6} genes")
    print("   ", batch_axis_check(Xc, batch, gene_lengths=gene_lengths))
    print()

{'TCGA': 84, 'CPTAC': 69} 

fibroblast    129 samples x  10853 genes
    {'var_explained': 0.124, 'participation_ratio': 4897, 'n_levels': 2, 'pc1_vs_batch_r': 0.0, 'abs_loading_vs_log_len_rho': 0.162, 'top_len_ratio': 1.91}

malignant     131 samples x   8585 genes
    {'var_explained': 0.164, 'participation_ratio': 4331, 'n_levels': 2, 'pc1_vs_batch_r': 0.0, 'abs_loading_vs_log_len_rho': 0.198, 'top_len_ratio': 1.33}

acinar         52 samples x   1236 genes
    {'var_explained': 0.207, 'participation_ratio': 732, 'n_levels': 2, 'pc1_vs_batch_r': 0.0, 'abs_loading_vs_log_len_rho': 0.371, 'top_len_ratio': 1.34}



The correction worked, and worked hard. `pc1_vs_batch_r` from 0.89 to **0.000** in all three — exactly 0 because cohort is now a column in `D`, so the residuals are orthogonal to it by construction. Same caveat as with θ: this is the identity, not a measurement. The meaningful evidence is elsewhere in the table.

**Variance explained collapsed.** 0.394 → 0.124 for fibroblast, 0.368 → 0.164 for malignant. So the batch axis was carrying two-thirds of what PC1 was, and it's gone.

**Participation ratio exploded.** 359 → 4,897 (of 10,853 genes) for fibroblast, 4,331 for malignant. The residual PC1 is now spread across nearly half the genes — that's the profile of unstructured noise, not a program. Which is the right outcome: after removing batch, no single dominant axis remains, so PC1 is just the largest slice of a diffuse remainder. Don't interpret it.

**Length bias is still there and worse in acinar.** ρ = 0.16/0.20/**0.37**, with top-length ratios of 1.9/1.3/1.3. The per-gene cohort shift absorbed the cohort *mean* difference but not the length structure — and acinar, with only 1,236 genes and 52 samples, is the most affected. Its participation ratio of 732 with ρ=0.37 means length still organizes a meaningful chunk of acinar's dominant axis.

**Two things to do.**

Rerun the cross-compartment correlations. That's the number the whole session hinged on:

```python
r, ns, ng = gene_corr(X_fib2, X_mal2)
print(f"fib x mal median r: {r.median():.3f}  ({ns} samples, {ng} genes)")
```

Previously 0.898. If it drops toward 0.5–0.6, the coupling was largely batch. If it stays near 0.9, something else is shared and the question reopens.

And check the θ side, which the correction doesn't touch:

```python
for ct in cmap.values():
    print(f"{ct:<22} {mc.df_theta[ct].reindex(batch.index).corr((batch=='TCGA').astype(float)):+.3f}")
```

Also note the sample counts moved: batch says 84/69 = 153, but the matrices have 129/131/52. The `min_theta` filter drops different samples per compartment, so per-cohort balance differs from what you printed. Worth checking acinar's split — 52 samples across two cohorts could be badly lopsided.

In [94]:
r, ns, ng = gene_corr(X_fib2, X_mal2)
print(f"fib x mal median r: {r.median():.3f}  ({ns} samples, {ng} genes)\n")

for ct in cmap.values():
    print(f"{ct:<22} {mc.df_theta[ct].reindex(batch.index).corr((batch=='TCGA').astype(float)):+.3f}")

fib x mal median r: 0.869  (121 samples, 5057 genes)

Ductal cell type 2     +0.305
Fibroblast cell        -0.181
Macrophage cell        -0.171
Endothelial cell       +0.049
Acinar cell            -0.215


Two results, and the first is the surprising one.

**fib×mal barely moved: 0.898 → 0.869.** Removing an axis that carried 38% of the variance and separated cohorts at r=0.89 cost only 0.03 of cross-compartment correlation. So batch was *not* what made the compartments correlate. The 0.87 was there underneath it and remains unexplained.

That means my closing account last time was wrong. Batch explained PC1 — it did not explain the coupling. Those were two separate phenomena and I collapsed them.

What's left to explain: two compartments correlating at 0.87 per gene, higher than either correlates with bulk (0.79/0.78), after θ, depth, and cohort are all residualized out. The bound argument still holds — a common cause can't produce correlation exceeding the paths to it — so whatever couples them is not any of those three and not bulk.

The remaining candidate I haven't tested is the deconvolution itself: `Z_full` allocates a shared read pool across compartments, so `Z[s,g,fib]` and `Z[s,g,mal]` are two partitions of the same `X_bulk[s,g]`. Within-compartment CPM removes the scale but not the fact that both derive from one number. That's structural to BayesPrism, present in fitted and unmodeled genes equally (0.912/0.898), and invisible in bulk PC1.

**θ carries residual cohort structure.** Ductal-2 at +0.31, acinar −0.22, fibroblast −0.18. Not large, but θ is upstream of `min_share`, `min_theta`, and every compartment matrix — and the post-hoc correction can't reach it. Ductal-2 at 0.31 means ~9% of malignant fraction variance is cohort.

That's an argument for per-cohort deconvolution rather than pooled-plus-correction, if the composition estimates matter to your conclusions.

Given how the session has gone, I'd flag that I've now proposed and retracted six or seven mechanisms. The reliable findings are: the fibroblast DE is circular (A=0, B=252), batch dominated PC1 and is now removed, and θ retains modest cohort structure. The 0.87 coupling I would leave as an open question rather than accept another explanation from me without a test that could falsify it.

In [95]:
# if you have pathologist purity estimates for either cohort
### print(mc.df_theta["Ductal cell type 2"].corr(purity_estimate))
# or compare within-cohort to between-cohort spread
t = mc.df_theta["Ductal cell type 2"]
print(t.groupby(batch).agg(["mean", "std", "count"]))

         mean    std  count
cohort                     
CPTAC   0.244  0.248     63
TCGA    0.413  0.274     82


In [96]:
print(t.groupby(batch).describe(percentiles=[.1,.25,.5,.75,.9]).round(3))
print("theta_mal < 0.05:", (t < 0.05).groupby(batch).sum().to_dict())

        count   mean    std  min    10%    25%    50%    75%    90%    max
cohort                                                                    
CPTAC    63.0  0.244  0.248  0.0  0.009  0.065  0.183  0.321  0.580  0.985
TCGA     82.0  0.413  0.274  0.0  0.067  0.207  0.371  0.590  0.806  1.000
theta_mal < 0.05: {'CPTAC': 12, 'TCGA': 7}


In [97]:
print(pd.crosstab(batch, [s.startswith("T-") for s in batch.index]))

col_0   False  True 
cohort              
CPTAC      20     49
TCGA        2     82


In [98]:
print((t > 0.9).groupby(batch).sum().to_dict())
print(t.nlargest(8).round(3).to_dict())
print()
print(pd.crosstab(batch, pd.Series([s.startswith("T-") for s in batch.index],
                                   index=batch.index)))

{'CPTAC': 4, 'TCGA': 5}
{'T-TCGA-FB-AAPP': 1.0, 'T-TCGA-US-A776': 1.0, 'N-C3L-04080': 0.985, 'N-C3L-00589': 0.984, 'T-TCGA-3A-A9IS': 0.974, 'T-TCGA-3A-A9IR': 0.971, 'T-TCGA-HZ-7289': 0.951, 'N-C3N-03173': 0.929}

col_0   False  True 
cohort              
CPTAC      20     49
TCGA        2     82


In [105]:
batch

T-C3L-02890       CPTAC
T-C3L-03635       CPTAC
T-C3L-02701       CPTAC
T-C3L-04072       CPTAC
T-C3L-00589       CPTAC
                  ...  
N-C3L-02606       CPTAC
N-C3N-03173       CPTAC
N-C3N-02696       CPTAC
N-TCGA-H6-8124     TCGA
N-TCGA-H6-A45N     TCGA
Name: cohort, Length: 153, dtype: object

In [108]:
print("Tumor")
tum = [s for s in t.index if s.startswith("T-")]
print(t.loc[tum].groupby(batch.loc[tum]).agg(["mean","std","count"]).round(3))

print("----"*10)

print("theta_mal vs cohort, tumors only:", round(t.loc[tum].corr((batch.loc[tum]=="TCGA").astype(float)), 3))

print("----"*10)

norm = [s for s in t.index if s.startswith("N-")]
print("Normal")
print(t.loc[norm].describe().round(3))
print(mc.df_theta.loc[norm, "Acinar cell"].describe().round(3))

Tumor
         mean    std  count
cohort                     
CPTAC   0.227  0.168     49
TCGA    0.417  0.274     81
----------------------------------------
theta_mal vs cohort, tumors only: 0.361
----------------------------------------
Normal
count    15.000
mean      0.292
std       0.417
min       0.000
25%       0.002
50%       0.066
75%       0.563
max       0.985
Name: Ductal cell type 2, dtype: float64
count    15.000
mean      0.381
std       0.447
min       0.000
25%       0.000
50%       0.034
75%       0.860
max       0.986
Name: Acinar cell, dtype: float64


In [111]:
nd = mc.df_theta.loc[norm, ["Ductal cell type 2","Acinar cell","Ductal cell type 1"]].round(3)
nd.sort_values("Ductal cell type 2")

,Ductal cell type 2,Acinar cell,Ductal cell type 1
N-C3L-04072,0.000,0.730,0.072
N-C3L-03123,0.000,0.985,0.003
N-C3N-00517,0.001,0.876,0.025
N-C3N-02765,0.002,0.986,0.000
N-C3L-07033,0.003,0.960,0.016
N-C3L-07037,0.012,0.844,0.061
N-C3L-00819,0.036,0.295,0.121
N-C3N-01899,0.066,0.000,0.036
N-TCGA-H6-8124,0.094,0.034,0.153
N-C3N-01719,0.136,0.000,0.000


Three distinct problems visible in that table.

**Vertex collapse, confirmed.** Four samples (N-C3L-01689, N-C3N-03173, N-C3L-00589, N-C3L-04080) sit at 0.91–0.985 Ductal-2 with acinar *exactly* 0.000. Not low — zero. A normal pancreas cannot have zero acinar content. That's the solver on a vertex, not an estimate. Six others are correctly acinar-dominant (0.73–0.986 acinar, ~0 Ductal-2), so the same pipeline handles most normals fine.

**A middle group that's collapsed in a third direction.** N-C3N-01899, N-C3N-01719, N-C3L-00640, N-TCGA-H6-8124: acinar 0.000 and Ductal-2 only 0.07–0.22. Their mass went somewhere else entirely — check where:

```python
mid = ["N-C3N-01899","N-C3N-01719","N-C3L-00640","N-TCGA-H6-8124","N-C3L-00819"]
print(mc.df_theta.loc[mid].round(3).T.to_string())
```

Exact zeros across multiple compartments in the same sample is the signature of the fixed point terminating at a simplex corner rather than converging.

**Seven samples are all-NaN.** N-C3L-07032, N-C3N-03069, N-C3N-02589, N-C3N-02996, N-C3L-02606, N-C3N-02696, N-TCGA-H6-A45N — no θ at all. They're in the index but the deconvolution produced nothing. Either they failed at input (gene overlap, zero library) or the sampler diverged. Worth knowing which, since if it's silent input failure it could affect tumors too:

```python
print(mc.df_theta.isna().all(axis=1).sum(), "samples with no theta")
print(df_bulk.loc[[s for s in mid_nan if s in df_bulk.index]].sum(axis=1))
```

**The count discrepancy explains earlier confusion:** you have 22 `N-` samples but only 15 with θ. Your groupby counted 145, the crosstab 153, the matrices 129/131/52. Those gaps are these NaNs plus the `min_theta` filter, and the differences have been quietly shifting the sample set between analyses all session.

Practical upshot: exclude normals from the tumor-state analyses. But the exact zeros are worth understanding first, because if vertex collapse happens in normals it can happen in low-purity tumors — and CPTAC tumors average θ=0.227 with 12 below 0.05.

In [116]:
mc.df_theta.head(3).T

,T-C3L-02890,T-C3L-03635,T-C3L-02701
Fibroblast cell,6.026e-01,7.597e-01,7.635e-01
Stellate cell,8.666e-03,5.456e-03,1.275e-08
Macrophage cell,2.504e-02,1.583e-02,2.703e-02
Endothelial cell,2.912e-02,2.786e-02,1.529e-02
T cell,8.828e-73,2.978e-163,1.957e-222
B cell,1.755e-02,8.385e-03,4.934e-03
Ductal cell type 2,1.051e-01,1.828e-01,1.751e-01
Endocrine cell,1.176e-152,4.146e-105,1.414e-02
Ductal cell type 1,1.083e-01,2.173e-72,1.685e-148
Acinar cell,1.036e-01,0.000e+00,0.000e+00


In [ ]:
tumor_samples = [s for s in mc.df_theta.index if s.startswith("T-")]
X_fib2 = mc.compartment_matrix(cmap["fibroblast"], min_share=MIN_SHARE, min_counts=MIN_COUNTS,
                               batch=batch, samples=tumor_samples)

z = (mc.df_theta.loc[tumor_samples] == 0).sum(axis=1)
print(z.value_counts().sort_index().to_dict())

mc.df_theta.loc[tumor_samples][z >= 3].T

T-C3L-02890    0
T-C3L-03635    1
T-C3L-02701    1
dtype: int64

### Do we have "purity_estimate"?

I wrote it as a placeholder for "some external purity estimate" — not a variable you have. Sorry, that was unclear.

Whether one exists depends on what's in your metadata:

```python
print([c for c in meta_desc.columns if any(k in c.lower() for k in
       ("purity", "tumor", "cellular", "neoplas", "percent", "pct"))])
```

**TCGA-PAAD** has several. The classic ones are pathologist estimates in the biospecimen slide data (`percent_tumor_nuclei`, `percent_stromal_cells`, `percent_normal_cells`) and computational estimates — ABSOLUTE purity from the Aran 2015 pan-cancer consensus table, ESTIMATE scores, or the TCGAbiolinks/`TCGAretriever` purity fields. If you pulled TCGA through GDC, the slide-level percentages may already be in your metadata.

**CPTAC3-PDAC** published tumor content estimates in the Cao 2021 supplementary tables, and CPTAC applied a purity threshold at accession — which is itself a reason the two cohorts could differ genuinely.

The catch: TCGA-PAAD is well known for low tumor purity with a wide spread, and pathologist estimates are notoriously noisy (inter-rater agreement is poor, and the section scored isn't the section sequenced). So a weak correlation between θ and a pathology estimate wouldn't cleanly indict θ.

A cheaper check that needs no external data — does the cohort difference in θ track a difference in the raw bulk signal?

```python
# epithelial marker burden in bulk, no deconvolution involved
epi = [<KRT19, KRT8, KRT18, EPCAM Ensembl IDs>]
e = bulk.loc[batch.index, [g for g in epi if g in bulk.columns]].mean(axis=1)
print(e.groupby(batch).agg(["mean", "std"]))
print("theta_mal vs bulk epithelial score:", round(mc.df_theta["Ductal cell type 2"].corr(e), 3))
```

If bulk epithelial content is also higher in TCGA, the θ difference is tracking something real in the data rather than being manufactured by the deconvolution.

### Is 9% like an error when comparing cohorts? What means the other 91% of the variance?

Not error — the 9% is only *explained by cohort*, and the 91% is everything else that makes tumors differ from each other.

Concretely, θ_Ductal-2 varies from sample to sample for many reasons. Cohort membership accounts for 9% of that spread. The remaining 91% is:

- **Real biological variation in tumor cellularity.** PDAC purity genuinely ranges from maybe 10% to 80%+ across patients — desmoplasia varies enormously. That's the dominant term and it's the signal you want.
- **Sampling variation.** Where the resection was taken, how much adjacent normal or stroma came with it.
- **Estimation error in θ itself.** BayesPrism's uncertainty, which your NNLS concordance table quantifies indirectly (Ductal-2 ρ=0.86 — good agreement on ordering, so error is modest for this compartment).

None of that is separated by the r² decomposition. All r² says is "cohort predicts 9% of it"; the other 91% is unpartitioned.

**Is 9% a problem?** It depends entirely on what you do with θ.

If you compare TCGA vs CPTAC3 directly — say, testing whether one cohort has more myCAF — then a 9% cohort component is a confound, because you can't tell composition differences from technical ones.

If you work within-cohort and meta-analyze, it's irrelevant. A between-group shift doesn't affect within-group correlations.

If you pool samples and correlate two θ-derived quantities, cohort acts as a lurking variable and can induce or mask correlation. That's the Simpson's-paradox risk, and it's why your per-cohort replication protocol matters.

For calibration: 9% is modest. A correlation of 0.31 on 153 samples is significant (p ≈ 0.0001) but small. The comparison worth making is to the batch effect in expression, where cohort explained ~89% of PC1's separation — two orders of magnitude more severe. θ is comparatively clean.

In [ ]:
cmap = {'malignant': 'Ductal cell type 2',
        'fibroblast': 'Fibroblast cell',
        'macrophage': 'Macrophage cell',
        'endothelial': 'Endothelial cell',
        'acinar': 'Acinar cell'
        }

In [ ]:
scores, cov = mc.program_scores(compartment_map=cmap, samples=tumor_samples)
print(scores.shape)
scores.head(3)

In [ ]:
cov.head(6)

In [ ]:
cov.r_with_theta.abs().max()

In [ ]:
mc.df_theta.head(3)

In [ ]:
theta = mc.df_theta

for col in scores.columns:
    ct = cmap[col.split(".")[0]]
    print(f"{col:<38} r {scores.loc[tumor_samples, col].corr(theta.loc[tumor_samples, ct]):+.3f}")

### Theta x Program-Compartments

So the scores are orthogonal to θ to machine precision, without decouple_purity. That doesn't happen by accident, and it isn't a biological result. Twenty-two programs across five compartments landing at 1e-17 means the orthogonality is enforced somewhere upstream — the scores are being computed on a quantity from which θ has already been projected out.

The likely site is the compartment matrix itmc. If Z[s,g,c] is normalized by θ[s,c] on construction — dividing the allocated counts by the fraction to get a per-cell expression estimate — then any score built from those values is θ-free by construction, and decouple_purity is a redundant flag operating one level too late.

### Two consequences

The compositional artifact I was worried about is ruled out. The malignant.prolif ↔ fibroblast.iCAF coupling cannot be θ_mal ↔ θ_fib leaking through, because neither score retains any θ component. That's a genuine strengthening of the result — and it's the one claim in your work that survives everything in this session.

But the same normalization is a candidate explanation for the 0.95 bulk correlation. Dividing by θ removes composition and leaves the per-sample bulk magnitude, which is shared across all compartments by construction. That would produce exactly what you observed: compartments that are θ-orthogonal and simultaneously near-identical to bulk. Reading compartment_matrix settles both questions at once.

In [ ]:
import inspect
# print(inspect.getsource(mc.compartment_matrix))

### Look for a division by θ, or by a row-derived scale factor. Also test it directly on the matrix rather than the scores:

In [ ]:
ct = cmap["fibroblast"]
t = theta.loc[X_fib2.index.intersection(theta.index), ct]
# theta x fibroblasts
r = X_fib2.loc[t.index].apply(lambda g: g.corr(t))
print(r.abs().max(), r.abs().median())

### No correlation betwee bulk and theta - scores correlates with bulk

Every gene is regressed on θ and log library size, and the fit is subtracted. So each column of logx is orthogonal to θ by construction. The 1e-17 correlations across all 22 programs are the residual-orthogonality identity, not a measurement. cov.r_with_theta is computed correctly and is structurally vacuous — it cannot report anything but zero as long as decouple=True, so it can't serve as a compositional-confounding check. Same failure mode as pc_theta_pearson, one level up.

This resolves the 0.95 bulk correlation. log(lib) is in the design matrix for every compartment separately. Residualizing on it removes each compartment's own library-size effect, but what remains is the gene's per-sample deviation from its own mean — and that deviation is dominated by the shared bulk signal, which was never a covariate. So the three compartments are each θ-free and each retain the same bulk component. Orthogonal to composition, near-identical to each other. Exactly what you measured.

And it explains min_share dropping 908 of 1604 fitted genes. The filter is (sh >= min_share).mean(axis=0) >= 0.5 — a gene must exceed 30% fibroblast share in at least half the samples. In tumors where fibroblast θ is low, even a canonical fibroblast marker fails that. The filter is selecting on composition, not on gene identity.

Two things this makes testable:

In [ ]:
X_fib_raw = mc.compartment_matrix(cmap["fibroblast"], min_share=0.3,
                                  min_counts=10, decouple=False)

ct = cmap["fibroblast"]
t = theta.loc[X_fib_raw.index.intersection(theta.index), ct]
# theta x fibroblasts
r = X_fib_raw.loc[t.index].apply(lambda g: g.corr(t))
print(r.abs().max(), r.abs().median())

### compartment_matrix

In [ ]:
mc.cell_types

In [ ]:
mc.Z_full.shape

In [ ]:
cell_name = 'Fibroblast cell'

cns = mc.build_CellNameState_from_full_Z(
            mc.Z_full, mc.genes_full, cell_types=mc.cell_types,
            cell_name=cell_name, samples=mc.df_theta.index,
            df_theta=mc.df_theta)

# samples x genes
cns.Z.head(3)

In [ ]:
mc.df_theta.head(3)

In [ ]:
Z = cns.Z
th = mc.df_theta[cell_name].reindex(Z.index)
th

In [ ]:
min_theta=0.02
low = th < min_theta
low.any()

In [ ]:
if low.any():
    print(
        f"{cell_name}: dropping {int(low.sum())} sample(s) with theta < "
        f"{min_theta} (min observed {th.min():.2e}); their compartment "
        "profile is prior, not signal.")
    Z = Z.loc[~low]
    th = th.loc[Z.index]

In [ ]:
Z

In [ ]:
Z.median(axis=0)

In [ ]:
min_counts=10

print(Z.shape)
Z = Z.loc[:, Z.median(axis=0) >= min_counts]
Z.shape

In [ ]:
lib = Z.sum(axis=1)
lib

In [ ]:
dead = ~np.isfinite(lib) | (lib <= 0)
if dead.any():
    print(
        f"{cell_name}: dropping {int(dead.sum())} sample(s) with zero "
        f"counts after filtering: {list(Z.index[dead])[:5]}")
    Z, lib, th = Z.loc[~dead], lib.loc[~dead], th.loc[Z.index[~dead]]

print(Z.shape, Z.index)

In [ ]:
cpm = Z.div(lib, axis=0) * 1e6
logx = np.log2(cpm + 1.0)
logx = logx.loc[:, np.isfinite(logx.values).all(axis=0)]
logx.shape

### if decouple

D = [1, th (theta cell type), lib (sum all genes = library)]

In [ ]:
D = np.column_stack([np.ones(len(th)), th.values, np.log(lib.values)])
D[:5]

In [ ]:
th = mc.df_theta[ct].reindex(Z.index)
th

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(th, np.log(lib), s=14, alpha=.7)
ax[0].set(xlabel=r"$\theta_{fib}$", ylabel="log(lib)")
ax[1].scatter(np.log(th), np.log(lib), s=14, alpha=.7)
ax[1].set(xlabel=r"log $\theta_{fib}$", ylabel="log(lib)")

D = np.column_stack([np.ones(len(th)), th.values, np.log(lib.values)])
print("rank", np.linalg.matrix_rank(D), "of", D.shape[1])
print("cond(D)", np.linalg.cond(D).round(1))
print("r(theta, log lib)     ", round(np.corrcoef(th, np.log(lib))[0, 1], 3))
print("r(log theta, log lib) ", round(np.corrcoef(np.log(th), np.log(lib))[0, 1], 3))
plt.tight_layout(); plt.show()

### Confirmed, and the magnitudes are informative in both directions.

The nonlinear dependence is real but small. Median |r| with log θ = 0.048, Spearman = 0.020, against 2.2e-15 for linear Pearson. So the residualization removes the linear component exactly and leaves a curvature residual of about 0.05. On ~112 samples that's well inside noise for any individual gene (|r| ≈ 0.19 is the p<0.05 threshold), so no gene is meaningfully θ-driven after decoupling.

In [ ]:
r_lin = X_fib2.loc[t.index].apply(lambda g: g.corr(t))            # ~1e-15
r_log = X_fib2.loc[t.index].apply(lambda g: g.corr(np.log(t)))    # ?
r_sp  = X_fib2.loc[t.index].apply(lambda g: g.corr(t, method="spearman"))
print(f"linear pearson: {r_lin.abs().median():.2e}, log: {r_log.abs().median():.2e}, spearman: {r_sp.abs().median():.2e}")

### But the median is the wrong statistic for a leakage question. What matters is the tail:

In [ ]:
fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)
list(fitted)[:5]

In [ ]:
r_log

In [ ]:
print(f"log-theta: median {r_log.abs().median():.3f}, "
      f"90th {r_log.abs().quantile(.9):.3f}, max {r_log.abs().max():.3f}, "
      f"n>0.3: {(r_log.abs() > 0.3).sum()}")
print(f"of those, fitted: {len(set(r_log.abs().nlargest(50).index) & fitted)}/50")

That closes it cleanly. Max |r| = 0.18 across all 10,853 genes, zero above 0.3, and the 90th percentile at 0.097. On ~112 samples the p<0.05 threshold is |r| ≈ 0.19 — so **not a single gene** shows significant residual dependence on log θ. The curvature I predicted from the left panel is there but negligible.

And 4/50 in the top tail is right at the 6.4% base rate for fitted genes, so even the weak residual isn't concentrated in the panel. No structure to it.

So: the decoupling works, the misspecification is harmless, and θ leakage is ruled out as an explanation for anything downstream. I was right that the linear column is the wrong functional form and wrong to imply it mattered.

**One thread left.** With θ eliminated, the 0.95 bulk correlation has to come from the projection — which is the split-by-class test, once `fitted` is back:


In [ ]:
bulk = np.log2(df_bulk.div(df_bulk.sum(axis=1), axis=0) * 1e6 + 1.0)

bulk.T.head(3)

In [ ]:
X_fib2.head(3)

In [ ]:
for name, gs in [("fitted", fitted), ("unmodeled", set(X_fib2.columns) - fitted)]:
    g = X_fib2.columns.intersection(list(gs))
    r, n_s, n_g = gene_corr(X_fib2[g], bulk.T)
    print(f"{name:<10} fib vs bulk  median r {r.median():.3f}  ({n_s} samples, {n_g} genes)")

Also worth resolving the `X` overlap oddity from the earlier run — fibroblast∩X was only 197 genes while malignant∩X was 2000. If `X` is a 2000-gene object rather than full bulk CPM, that 0.956 was computed on a small non-random subset and the number needs redoing before it carries weight.

**Where the session stands.** Two solid negatives: the fibroblast gene-level DE is circular (A=0, B=252, B fibroblast-derived), and the compartment matrices carry a large shared component that gene-level cross-compartment analysis can't see past. Two solid positives: the decoupling is doing real work (median |r| 0.20 → 2e-15), and rank-concordance with an independent NNLS estimator is high for fibroblast, acinar, and Ductal-2.

Two library items worth filing while they're fresh: `cond(D)` alongside the existing rank check, and `cov.r_with_theta` gated on `decouple` so it stops reading as a passed check.

Full overlap this time — 696 and 10,157, exactly the class sizes, on 129 samples. So the gene universes match and the earlier 197/2000 was `X` being the wrong object.

**This overturns the 0.95 claim.** Real bulk gives 0.79 and 0.72, not 0.956. I built "the compartments are the bulk wearing labels" on a 197-gene correlation against an object I never checked. That conclusion was wrong, and the numbers here don't support it: at r ≈ 0.75, roughly 44% of variance is shared with bulk and the rest is not. The compartment matrices carry substantial independent between-sample variation.

**And the class ordering is now the right way round.** Fitted 0.790 > unmodeled 0.723 — the projected genes are *less* bulk-like than the fitted ones, which is the opposite of what the `Z ∝ bulk × θ × φ` argument predicts. So the projection isn't imposing the bulk signal either. Both of my mechanistic explanations for the shared component are dead, and the 0.067 gap goes in a direction neither predicts.

One caution before reading anything into that gap: it's 696 genes vs 10,157, and fitted genes are Peng panel markers — high-expressing, high-variance, cell-type-specific. Those correlate better with bulk for ordinary reasons. A bootstrap CI would tell you whether 0.067 is more than class-composition noise, but I'd not lean on it.

**What still needs explaining** is the fib×mal 0.898, since it's now *higher* than either compartment's correlation with bulk. Two variables sharing a common factor can't correlate more with each other than each does with the factor — so bulk is not the shared component. Something else couples the compartments, and PC1 concordance of 0.935 says it's one dominant axis.

The candidate worth testing is the reference itself: both compartments' unmodeled genes are reconstructed with the same φ, so a shared reference-driven structure would appear in both without appearing in bulk.

In [ ]:
for name, gs in [("fitted", fitted), ("unmodeled", set(X_fib2.columns) - fitted)]:
    g = X_fib2.columns.intersection(list(gs)).intersection(X_mal2.columns)
    r, n_s, n_g = gene_corr(X_fib2[g], X_mal2[g])
    print(f"{name:<10} fib x mal  median r {r.median():.3f}  ({n_s} samples, {n_g} genes)")

If unmodeled genes couple at 0.95 while fitted sit near 0.6, the coupling is in the projection. If both are ~0.9, it's shared biology or something upstream in `full_Z` common to both classes.

Worth saying plainly: three of the diagnostics I proposed in this session pointed at mechanisms that turned out not to be operating, and the one number I built the strongest claim on was computed against an unverified object. The circularity finding (A=0, B=252) is the result that has held up.

Both classes at ~0.90, so the coupling isn't in the projection. Combined with the bulk numbers, the picture is now internally contradictory in a way that points somewhere specific.

fib↔bulk 0.79, mal↔bulk (needs rerunning against real bulk, was 0.946 against the wrong object), fib↔mal 0.898. If bulk were the shared factor, fib↔mal couldn't exceed fib↔bulk — correlation through a common cause is bounded by the product of the paths. 0.898 > 0.790 means the two compartments share something bulk doesn't carry.

Two candidates, and one test separates them.

**Shared reference φ.** Both compartments' values derive from the same Peng reference through `full_Z`. But this should hit unmodeled genes harder than fitted ones, and it doesn't — 0.898 vs 0.912.

**The zero-sum constraint.** θ across cell types sums to 1, so `Z_full` allocation for one compartment is mechanically tied to the others. That's upstream of both classes and would affect them equally, which matches what you see. It's also the thing your `min_theta` docstring is already worried about.

The discriminating run — malignant against real bulk, split the same way:

```python
for name, gs in [("fitted", fitted), ("unmodeled", set(X_mal2.columns) - fitted)]:
    g = X_mal2.columns.intersection(list(gs))
    r, n_s, n_g = gene_corr(X_mal2[g], bulk.T)
    print(f"{name:<10} mal vs bulk  median r {r.median():.3f}  ({n_s} samples, {n_g} genes)")
```

Note `fitted` there is the *fibroblast* panel — for the malignant matrix you want `set(res.cell_type_expression(cmap["malignant"]).index)`. Using the fibroblast set would make the split meaningless.

If mal↔bulk also lands ~0.75–0.80 while fib↔mal is 0.898, the excess coupling is real and compartment-specific, and the zero-sum route is the one to chase. Two things would test it: whether the residual coupling correlates with θ_fib + θ_mal (their sum is what the constraint acts on), and whether it survives a CLR transform of θ before deconvolution.

The n=34 fitted class is still too small to read — the 0.912 has a CI of roughly ±0.026 from the earlier bootstrap, so treat it as indistinguishable from the unmodeled 0.898.

In [ ]:
for name, gs in [("fitted", fitted), ("unmodeled", set(X_mal2.columns) - fitted)]:
    g = X_mal2.columns.intersection(list(gs))
    r, n_s, n_g = gene_corr(X_mal2[g], bulk.T)
    print(f"{name:<10} mal vs bulk  median r {r.median():.3f}  ({n_s} samples, {n_g} genes)")

Note the `fitted` here is still the fibroblast panel — 500 of the 696 fibroblast panel genes happen to be in `X_mal2.columns`. So this is "fibroblast panel genes measured in the malignant compartment," not the malignant panel. The split doesn't mean what the label says. Rerun with `set(res.cell_type_expression(cmap["malignant"]).index)` before reading anything into the 0.834/0.770 gap.

The medians themselves are still usable, since both classes land in the same range: **mal↔bulk ≈ 0.78, fib↔bulk ≈ 0.73–0.79, fib↔mal = 0.898.**

That inequality is the finding. If bulk were the common cause, fib↔mal would be bounded near 0.79 × 0.78 ≈ 0.62. Observing 0.898 means the two compartments share a component that bulk does not carry — roughly 0.28 of correlation beyond what the shared bulk signal can explain.

That's a strong constraint. Whatever couples the compartments is introduced *by the deconvolution*, not inherited from the input. It's present in fitted and unmodeled genes equally (0.912 vs 0.898), so it's upstream of the panel/projection split — which puts it in `Z_full` allocation itself.

The zero-sum constraint fits all three observations. Under `θ_fib + θ_mal + ... = 1`, allocating reads to one compartment mechanically constrains the others, and the resulting anticorrelation in allocation shows up as a shared per-sample factor once each matrix is CPM-normalized within itself.

Two tests:

In [ ]:
# 1. Does the residual coupling track the compositional pair?
idx = X_fib2.index.intersection(X_mal2.index)
g = X_fib2.columns.intersection(X_mal2.columns)

print(f"Common samples: {len(idx)} and genes: {len(g)}")

print("")

pf, _ = pc1(X_fib2.loc[idx, g])
pm, _ = pc1(X_mal2.loc[idx, g])

tf = theta.loc[idx, cmap["fibroblast"]]
tm = theta.loc[idx, cmap["malignant"]]

print("PC1_fib vs PC1_mal:", round(pf.corr(pm), 3))
print("theta_fib vs theta_mal:", round(tf.corr(tm), 3))

print("")

print("PC1_fib vs theta_fib+theta_mal:", round(pf.corr(tf + tm), 3))
print("PC1_mal vs theta_fib+theta_mal:", round(pm.corr(tf + tm), 3))

print("")

print("PC1_fib vs theta_fib:", round(pf.corr(tf), 3))
print("PC1_mal vs theta_mal:", round(pm.corr(tm), 3))


# 2. Partial correlation of fib x mal given bulk, per gene.

If `θ_fib ↔ θ_mal` is strongly negative and the PC1s track their sum, that's the mechanism — and it's the same zero-sum concern that motivated your CLR work. The second test is the direct version: if fib↔mal drops toward 0.6 after partialling out the bulk value for each gene, the excess is entirely the deconvolution's doing.

The zero-sum hypothesis is dead. PC1_fib↔θ_fib = −0.12, PC1_mal↔θ_mal = +0.10, and the sum term at 0.19/−0.12. None of that supports a compositional mechanism — the shared axis is essentially orthogonal to θ, which is what the residualization guarantees anyway. I proposed a mechanism that the decoupling had already ruled out; I should have seen that before suggesting it.

So: PC1_fib↔PC1_mal = 0.935, uncorrelated with θ in either compartment, present equally in fitted and unmodeled genes, and stronger than either compartment's correlation with bulk. Every mechanical explanation I've offered this session has failed a test.

**θ_fib ↔ θ_mal = −0.53 is worth separating out.** That's the composition, not the expression, and it's a real negative — more fibroblast means less malignant, which is what desmoplasia looks like. It's also, notably, in the same range as your replicated prolif↔iCAF coupling (−0.46 to −0.52). Since the program scores are θ-orthogonal by construction, those can't be the same number, but the coincidence in magnitude is close enough that I'd want to see them plotted against each other before treating them as independent findings.

**What's left to check on the 0.935.** The residualization subtracts `D @ beta` and adds back `logx.mean(axis=0)`. Both compartments get the same treatment on the same samples with a near-collinear design (cond ≈ 430 by the simulation). If `lstsq` is leaving a similar structured residual in both — the depth component that θ and log(lib) couldn't separate — that would be shared, θ-orthogonal, class-independent, and absent from bulk. All four observations.

```python
# what does PC1 actually track?
print("PC1_fib vs log(lib_fib):", round(pf.corr(np.log(lib_fib.loc[idx])), 3))
print("PC1_fib vs bulk PC1:", ...)   # bulk PC1 on the same samples/genes
top = pd.Series(np.abs(pc1_loadings_fib), index=g).nlargest(30)
print(top.index.tolist())   # what genes carry it?
```

The loadings are the most direct route. If the top 30 are mitochondrial, ribosomal, or hemoglobin, it's technical. If they're a coherent biological program, the compartments genuinely co-vary and the whole line of inquiry resolves in your favor.

### what does PC1 actually track?


In [ ]:
def compartment_lib(mc, cell_name, min_share=0.3, min_counts=10, min_theta=0.02):
    cns = mc.build_CellNameState_from_full_Z(
        mc.Z_full, mc.genes_full, cell_types=mc.cell_types,
        cell_name=cell_name, samples=mc.df_theta.index, df_theta=mc.df_theta)
    Z = cns.Z
    th = mc.df_theta[cell_name].reindex(Z.index)
    Z = Z.loc[th >= min_theta]
    Z = Z.loc[:, Z.median(axis=0) >= min_counts]
    if min_share > 0 and cns.share is not None:
        sh = cns.share.reindex(index=Z.index, columns=Z.columns)
        Z = Z.loc[:, (sh >= min_share).mean(axis=0) >= 0.5]
    return Z.sum(axis=1)

lib_fib = compartment_lib(mc, cmap["fibroblast"])
lib_mal = compartment_lib(mc, cmap["malignant"])
assert lib_fib.index.equals(X_fib2.index)   # should match if filters replicate
assert lib_mal.index.equals(X_mal2.index)  

In [ ]:
print("PC1_fib vs log(lib_fib):", round(np.log(pf).corr(np.log(lib_fib.loc[idx])), 3))

In [ ]:
print("PC1_fib vs log(lib_fib):", round(pf.corr(np.log(lib_fib.loc[idx])), 3))

In [ ]:
pc1_bulk = pc1(bulk.T.loc[idx, g])
pc1_bulk

In [ ]:
def pc1_full(M):
    """Sample scores, variance explained, and gene loadings."""
    Z = ((M - M.mean()) / M.std()).fillna(0.0)
    A = Z.values - Z.values.mean(0)
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    scores = pd.Series(U[:, 0] * S[0], index=M.index)
    loadings = pd.Series(Vt[0], index=M.columns)
    return scores, (S**2 / (S**2).sum())[0], loadings

pf, vf, load_fib = pc1_full(X_fib2.loc[idx, g])
top = load_fib.abs().nlargest(30)
print(load_fib[top.index].round(3).to_dict())

In [ ]:
lib_fib = compartment_lib(mc, cmap["fibroblast"])
lib_mal = compartment_lib(mc, cmap["malignant"])

bulk_f = bulk.T.loc[idx, g]
pc1_bulk, var_bulk = pc1(bulk.T.loc[idx, g])

g = X_fib2.columns.intersection(list(fitted))
r, n_s, n_g = gene_corr(X_fib2[g], bulk.T)

print("PC1_fib vs log(lib_fib):", round(pf.corr(np.log(lib_fib.loc[idx])), 3))
print("PC1_fib vs bulk PC1:", round(pf.corr(pc1_bulk), 3))
dic = load_fib[top.index].round(3).to_dict()
for gene, value in dic.items():
    print(f"{gene}: {value}")

In [ ]:
print("participation ratio:", round(1 / (load_fib**4).sum()))   # ~effective n genes
print("top 100 share:", round((load_fib.abs().nlargest(100)**2).sum(), 3))

Map the symbols before interpreting. I recognize ENSG00000108821 as COL1A1 and think ENSG00000117122 is MFAP2, both loading positive while most of the top 30 are negative. If the positive tail is ECM/collagen and the negative tail is something else, that's a desmoplasia axis and a real biological finding. But I'd be guessing on 28 of 30, so:

In [ ]:
gene_map = mc.prism.load_gene_map("geneid")
gene_map.head(3)

In [ ]:
gene_map.reindex(load_fib.abs().nlargest(40).index).head(4)

In [ ]:
N = len(top.index)

dftop = gene_map.reindex(load_fib.abs().nlargest(N).index).copy()

dftop["loading"] = load_fib.reindex(top.index).round(3)
dftop

Not tumor-vs-normal. No digestive enzymes anywhere — so my acinar prediction was wrong and the axis is something else.

The negative pole splits into two recognizable groups:

- **Lymphocyte:** ITGA4, CD226, THEMIS, STAT4, KCNA3, SLFN12L, SAMD3 — T/NK-restricted, several of them (THEMIS, CD226, STAT4) essentially T-cell-specific.
- **Quiescent/normal stromal:** ABCA6, ABCA10, ABCC9, ABCD2, HMCN1, LAMA2, PREX2 — the ABCA6/8/9/10 cluster is a known resting-fibroblast signature, ABCC9 is pericyte, LAMA2 is basement membrane.

Positive pole: COL1A1, MFAP2, CERCAM, EHD2, TMEM158 — activated matrix production. TMEM158 is Ras-induced.

So PC1 is **immune-infiltrated, quiescent stroma at one end; collagen-producing activated stroma at the other.** That's a desmoplasia/immune-exclusion axis, and it's the same biology as your quiescent fibroblast identity-loss signature.

Two things follow, and one is a problem.

**It's not an artifact.** Participation ratio 359, orthogonal to bulk PC1 (0.244), to θ (−0.12), to log lib (−0.14), and it has a coherent gene-level interpretation. The 0.935 fib↔mal concordance is two compartments both reporting desmoplasia, which they genuinely should. Nothing in this session's earlier "compartments are redundant" framing survives — that was built on the 0.956 against the wrong `X`.

**But the immune genes shouldn't be here.** ITGA4, CD226, THEMIS, STAT4 are lymphocyte genes loading on the *fibroblast* compartment's first principal component. Those are the compartments where NNLS concordance collapsed (T cell ρ=0.13, macrophage ρ=0.12, B cell ρ=0.17, T-cell mean θ = 5.4e-4). So immune expression that BayesPrism couldn't allocate to a near-empty T-cell compartment appears to be landing in fibroblast instead — and it's strong enough to define PC1.

That's testable directly:

```python
imm = ["ENSG00000115232","ENSG00000150637","ENSG00000172673","ENSG00000138378"]  # ITGA4,CD226,THEMIS,STAT4
for c in ["fibroblast","malignant"]:
    Xc = X_fib2 if c=="fibroblast" else X_mal2
    print(c, Xc[[i for i in imm if i in Xc.columns]].corrwith(
        theta.loc[Xc.index, "T cell"]).round(3).to_dict())
```

If fibroblast T-cell gene expression tracks θ_T, the allocation is at least directionally right. If it doesn't, the immune signal is misallocated — which would also explain the fib↔mal coupling, since misallocated immune reads would land in both compartments simultaneously.

Worth running the same PC1 loadings on `X_mal2`. If malignant PC1 carries the same lymphocyte genes, that's shared misallocation rather than shared biology.

In [ ]:
imm = ["ENSG00000115232","ENSG00000150637","ENSG00000172673","ENSG00000138378"]  # ITGA4,CD226,THEMIS,STAT4
for c in ["fibroblast","malignant"]:
    Xc = X_fib2 if c=="fibroblast" else X_mal2
    print(c, Xc[[i for i in imm if i in Xc.columns]].corrwith(
        theta.loc[Xc.index, "T cell"]).round(3).to_dict())

r = 0.04–0.17 against θ_T. On 129 samples the p<0.05 threshold is |r| ≈ 0.17, so STAT4 is borderline and the rest are noise. T-cell gene expression in the fibroblast compartment does **not** track the T-cell fraction.

Two readings, and they have different consequences.

**Misallocation.** T-cell reads are being assigned to fibroblast rather than to the T-cell compartment. Consistent with θ_T mean = 5.4e-4 (essentially zero mass available to receive them) and NNLS concordance ρ = 0.13. If so, THEMIS/CD226/STAT4 variation in `X_fib2` is lymphocyte infiltration wearing a fibroblast label, and it's strong enough to define PC1.

**θ_T is unusable, so the test is uninformative.** At mean 5.4e-4 with rank concordance 0.13, θ_T may simply not be measuring T-cell content at all. Then a near-zero correlation says nothing about where the reads went — you'd be correlating against noise.

These aren't distinguishable from this number alone. What separates them is an external estimate of T-cell content:

```python
# T-cell markers in the BULK, which needs no deconvolution to be meaningful
tcell = [<CD3D, CD3E, CD2, IL7R, CD8A Ensembl IDs>]
t_bulk = bulk.loc[X_fib2.index, [i for i in tcell if i in bulk.columns]].mean(axis=1)

print("theta_T vs bulk T score:", round(theta.loc[X_fib2.index,"T cell"].corr(t_bulk), 3))
print("PC1_fib vs bulk T score:", round(pf.corr(t_bulk.loc[idx]), 3))
```

The first line diagnoses θ_T. If it's near zero, θ_T is broken and should be dropped from the analysis entirely — which also invalidates the macrophage/endothelial program scores built on similarly unreproducible θ.

The second is the one that matters: if fibroblast PC1 tracks bulk T-cell content at |r| > 0.5, then PC1 is carrying immune infiltration regardless of what θ_T says, and the fibroblast compartment is not compartment-pure.

That would also explain the 0.935 — immune infiltration misallocated into both fibroblast and malignant would appear as shared variation invisible to bulk PC1. Running the same loadings on `X_mal2` tests it from the other side: same lymphocyte genes in malignant PC1 means shared leakage, not shared biology.

In [ ]:
gene_map = prism.load_gene_map("geneid")
gene_map.reset_index(inplace=True)
gene_map.set_index("symbol", inplace=True)
gene_map

In [ ]:
bulk.head(2)

In [ ]:
[x for x in tcell if x in bulk.index.to_list()]

In [ ]:
# T-cell markers in the BULK, which needs no deconvolution to be meaningful
tcell_list = ['CD3D', 'CD3E', 'CD2', 'IL7R', 'CD8A']

tcell = [gene_map.loc[x, "geneid"] for x in tcell_list]  # Convert Ensembl IDs to symbols

t_bulk = bulk.loc[[x for x in tcell if x in bulk.index.to_list()]].mean(axis=0)

t_bulk

In [ ]:
theta.loc[X_fib2.index,"T cell"]

In [ ]:
print("theta_T vs bulk T score:", round(theta.loc[X_fib2.index,"T cell"].corr(t_bulk), 3))
print("PC1_fib vs bulk T score:", round(pf.corr(t_bulk.loc[idx]), 3))

Both numbers are informative, and together they contradict the misallocation hypothesis.

**θ_T vs bulk T score = 0.386.** So θ_T isn't broken. It's weak — 15% of variance, and low for something as directly measurable as T-cell content from CD3D/CD3E in bulk — but it's tracking real T-cell abundance. That partially rehabilitates the T-cell compartment, though ρ=0.13 against NNLS still says the two estimators disagree badly on ordering.

**PC1_fib vs bulk T score = −0.013.** This kills the leakage story. If lymphocyte reads were being misallocated into fibroblast strongly enough to define PC1, PC1 would track bulk T-cell content. It doesn't — zero, on 121 samples.

So ITGA4, CD226, THEMIS, STAT4 load on fibroblast PC1 while that component is orthogonal to actual T-cell infiltration. My misallocation hypothesis was wrong, and that's now four mechanical explanations I've proposed and had falsified in this session.

**What the axis is instead.** With immune abundance ruled out, the negative pole reads differently: ABCA6/ABCA8/ABCA10/ABCC9/LAMA2/HMCN1 is the quiescent-stroma signature, and those immune genes may be co-varying with it rather than driving it — normal pancreatic stroma and resident lymphocytes both being present in low-desmoplasia tissue. The positive pole (COL1A1, MFAP2, CERCAM) is unambiguous activated matrix.

That is your quiescent fibroblast identity-loss axis, appearing as the dominant component of the fibroblast compartment, shared with malignant at 0.935, and invisible to bulk PC1. It's the one thing in this session that has survived every test I threw at it.

Two things worth doing before trusting it:

If malignant PC1 shows an epithelial program rather than these same stromal genes, the 0.935 is two compartments independently reporting the same tumor state — which is a finding. If it shows COL1A1 and ABCA6 again, the two matrices are sharing genes they shouldn't and that needs explaining before anything else.

In [ ]:
idx

In [ ]:
g_common = X_fib2.columns.intersection(X_mal2.columns)   # 5057 genes
len(g_common)

In [ ]:
pm_s, vm, load_mal = pc1_full(X_mal2.loc[idx, g_common])

In [ ]:
load_mal.abs().nlargest(30).index

In [ ]:
gene_map.head(2)

In [ ]:
gene_map = prism.load_gene_map("geneid")
gene_map.reindex(load_mal.abs().nlargest(30).index)

In [ ]:
# 1. Same loadings on malignant. Shared biology or shared artifact?
pm_s, vm, load_mal = pc1_full(X_mal2.loc[idx, g_common])
lista = gene_map.reindex(load_mal.abs().nlargest(30).index)["symbol"].tolist()
print("; ".join(lista))

# 2. Does PC1 track the myCAF/iCAF axis score?
print(round(pf.corr(scores.loc[idx.intersection(scores.index), "fibroblast.axis_myCAF_minus_iCAF"]), 3))

That's not a biological program. BIRC6, CEP350, VPS13C, KIAA1109, DST, SMG1, AHCTF1, USP34, BDP1 — these are among the longest transcripts in the genome, and the list has no functional coherence whatsoever. Ubiquitin ligases, kinesins, spliceosome components, nuclear pore, NF1. Nothing connects them except **transcript length**.

That's a technical axis: 3′ bias, RNA degradation, or coverage-uniformity differences between samples. Long transcripts lose proportionally more signal when RNA is degraded, so they co-vary as a block. In a cohort mixing TCGA and CPTAC3 with different library preps — and given the strandedness issue you already found — this is exactly the kind of thing that would show up.

So the malignant compartment's dominant axis is technical, while the fibroblast compartment's is biological (quiescent vs activated stroma). Two completely different components.

**Which makes the 0.935 hard to interpret.** Those are computed on the same 5057 genes and the same 121 samples, yet carry unrelated gene loadings. PC1s with orthogonal-looking loadings shouldn't correlate at 0.935 in sample space. Something is off — either the SVD sign/component indexing, or PC1_mal and PC1_fib aren't the objects I think they are.

Check that the numbers still reconcile:

```python
pf2, vf2, load_fib2 = pc1_full(X_fib2.loc[idx, g_common])
pm2, vm2, load_mal2 = pc1_full(X_mal2.loc[idx, g_common])
print(round(abs(pf2.corr(pm2)), 3), round(vf2, 3), round(vm2, 3))
print(round(abs(np.corrcoef(load_fib2[g_common], load_mal2[g_common])[0,1]), 3))
```

If the score correlation is 0.935 but the loading correlation is near zero, that's mathematically strange on identical features and needs explaining before anything else. Also verify `pf` in your session is still from `pc1_full` on `g_common` and not the earlier `pc1` call on a different gene set — `g` was reassigned mid-session, which is precisely how this kind of mismatch happens.

The 0.086 (PC1_fib vs the myCAF−iCAF axis score) is worth noting separately: the fibroblast axis you've been running DE on is essentially unrelated to the fibroblast compartment's dominant variation.

In [ ]:
pf2, vf2, load_fib2 = pc1_full(X_fib2.loc[idx, g_common])
pm2, vm2, load_mal2 = pc1_full(X_mal2.loc[idx, g_common])
print(round(abs(pf2.corr(pm2)), 3), round(vf2, 3), round(vm2, 3))
print(round(abs(np.corrcoef(load_fib2[g_common], load_mal2[g_common])[0,1]), 3))

Loading correlation 0.996. The two PC1s are the *same gene direction* — so my read of the malignant list as "unrelated to fibroblast" was wrong, and it was wrong because I only looked at the top 30 of each. At participation ratio 359, the top 30 is under 10% of the component; two vectors can agree at 0.996 overall while their extreme tails differ.

So there is one axis, shared between compartments at r=0.996 in gene space, explaining ~38% of variance in each. The question is what it is — and the two tails now have to be reconciled rather than treated as separate answers.

The malignant tail is length-confounded (BIRC6, KIAA1109, VPS13C, DST, SMG1, AHCTF1 — no functional coherence, all very long). The fibroblast tail is stromal-coherent (ABCA6/10, LAMA2, HMCN1, COL1A1). Both are the top of the same vector, which means the axis has a technical component and a biological one entangled, and which genes surface at the extreme depends on the compartment's noise structure.

Transcript length is the testable half:

```python
# gene length from your annotation
L = gene_lengths.reindex(g_common)          # or biomart/GTF-derived
print("loading vs log length:", round(load_fib2.corr(np.log(L), method="spearman"), 3))
print("|loading| vs log length:", round(load_fib2.abs().corr(np.log(L), method="spearman"), 3))
```

If |r| > 0.4, length is driving a substantial part of the component and the compartment matrices inherit a coverage/degradation artifact that CPM normalization doesn't remove. Given your TCGA/CPTAC3 mix and the strandedness history, worth also checking:

```python
cohort = pd.Series(["TCGA" if "TCGA" in s else "CPTAC" for s in idx], index=idx)
print(round(pf2.corr((cohort=="TCGA").astype(float)), 3))
```

Cohort separation on PC1 would explain the 0.996 immediately — batch is shared across all compartments by construction, θ-orthogonal, bulk-PC1-orthogonal (if bulk was normalized differently), and concentrated in a few hundred length-sensitive genes.

That single line is the cheapest remaining test, and if it comes back high, the desmoplasia reading of the fibroblast loadings was me pattern-matching on a tail.

In [ ]:
pf2

In [ ]:
load_fib2

In [ ]:
load_fib2[g_common]

In [ ]:
load

In [ ]:
gene_len

In [ ]:
gene_lengths = pd.Series(gene_len, name="exonic_len")
gene_span = pd.Series(span, name="genomic_span")
print(len(gene_lengths), gene_lengths.reindex(g_common).notna().mean().round(3))

In [ ]:
top_mal = load_mal2.abs().nlargest(30).index

L = np.log(gene_lengths.reindex(g_common).dropna())
lm = load_mal2.reindex(L.index)
lf = load_fib2.reindex(L.index)

print("mal |loading| vs log exonic len:", round(lm.abs().corr(L, method="spearman"), 3))
print("fib |loading| vs log exonic len:", round(lf.abs().corr(L, method="spearman"), 3))
print("top30 median len:", int(np.exp(L.reindex(top_mal).dropna().median())),
      " background median:", int(np.exp(L.median())))

Two numbers pointing in different directions, and the resolution matters.

**The top-30 is length-biased.** Median 10,830 bp vs background 4,685 — 2.3×, and that's after the annotation covers 99.9% of your genes so it isn't a mapping artifact. The malignant top-30 list I called "no functional coherence" is confirmed as a long-gene set, and the GO enrichment you ran on it is almost certainly the shadow of that.

**But length doesn't drive the component.** Spearman 0.21/0.24 over all 5057 genes — about 5% of variance. So length is concentrated in the extreme tail while the bulk of the 359-gene effective set isn't length-driven. Both things are true: the tail I read symbols off is contaminated, and the axis as a whole is not a length artifact.

Which means reading the top 30 was the wrong move on both compartments. At participation ratio 359, the top 30 is <10% of the component and it's precisely the part where length bias concentrates. My desmoplasia interpretation of the fibroblast tail (COL1A1, ABCA6, LAMA2) has the same problem — those loadings sit in the same contaminated region.

The way to see the axis without the tail:

In [ ]:
# length-residualized loadings
Lz = (L - L.mean()) / L.std()
lf_r = lf.reindex(L.index)
lf_res = lf_r - Lz * (lf_r.corr(Lz) * lf_r.std())
top_res = lf_res.abs().nlargest(40).index
lista = gene_map.reindex(top_res)["symbol"].tolist()
lista.sort()

np.array(lista)


If ABCA6/LAMA2/COL1A1 survive that, the desmoplasia reading holds. If a different set surfaces, the tail was length and the axis is something else.

**Still unrun, five requests in, and it's one line:**



Cohort would explain a 38%-variance axis shared across compartments at r=0.996 in gene space, orthogonal to θ and to bulk PC1, with a length-biased tail — TCGA and CPTAC3 differ in library prep, and you've already found one strandedness artifact between them. It's the single most likely explanation left and it costs nothing to rule out.

In [ ]:
corr = round(pf2.corr(pd.Series([("TCGA" in s) for s in idx], index=idx).astype(float)), 3)
corr

In [ ]:
idx

In [ ]:
sert = pd.Series([("TCGA" in s) for s in idx], index=idx).astype(float)
sert.index

In [ ]:
sert

In [ ]:
pf2

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
for i, (lab, sub) in enumerate([("CPTAC", pf2[sert == 0]), ("TCGA", pf2[sert == 1])]):
    ax.scatter(sub, np.full(len(sub), i) + np.random.uniform(-.12, .12, len(sub)),
               s=16, alpha=.6, label=f"{lab} (n={len(sub)})")
ax.set_yticks([0, 1], ["CPTAC", "TCGA"])
ax.set_xlabel("PC1_fib")
ax.set_title(f"point-biserial r = {corr:.3f}")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
disc = mc.discretize_axes(scores)

disc.head(6)

In [ ]:
R_fib = mc.factorial_state_de(X_fib2, disc)
R_mal = mc.factorial_state_de(X_mal2, disc)
R_aci = mc.factorial_state_de(X_aci2, disc)
 
R_fib.head(3)

In [ ]:
cmap["fibroblast"]

In [ ]:
fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)
len(fitted), list(fitted)[:5]

In [ ]:
background = set(R_fib.index)

In [ ]:
print(f"\nfibroblast background ({len(background)} genes): "
      f"fitted {len(fitted & background)} "
      f"({100 * len(fitted & background) / len(background):.1f}%), "
      f"unmodeled {len(background - fitted)}")
print("\n" + "-" * 68 + "\n")

### program_scores

In [ ]:
import warnings

In [ ]:
all_genes = [g for progs in mc.PROGRAMS.values()
                for genes in progs.values() for g in genes]

len(all_genes), all_genes[:5]

In [ ]:
gene_map = mc.prism.load_gene_map("geneid")
gene_map.head(3)

In [ ]:
sym2id = {s: i for i, s in gene_map["symbol"].astype(str).items()}

i = 0
for symbol, geneid in sym2id.items():
    print(symbol, geneid)
    i += 1
    if i >= 5:  # Print only the first 5 items
        break

In [ ]:
compartment_map=None

# Majority vote over every gene, not the first one of an arbitrary
# compartment: dict order is not meaningful and one stray entry should
# not decide the vocabulary for all of them.
n_ensg = sum(g.startswith("ENSG") for g in all_genes)
translate = n_ensg < len(all_genes) / 2

if 0 < n_ensg < len(all_genes):
    warnings.warn(
        f"PROGRAMS mixes vocabularies: {n_ensg}/{len(all_genes)} look "
        "like ensembl ids. Translating the symbols only.")

programs = mc.PROGRAMS
sym_missing = {}
if translate:
    # NOT written back to mc.PROGRAMS: that made the transform
    # destructive -- a translation that silently produced empty lists
    # became its own input on the next call and could not be recovered.
    programs = {
        comp: {prog: [sym2id.get(g, g) if not g.startswith("ENSG") else g
                        for g in genes
                        if g.startswith("ENSG") or g in sym2id]
                for prog, genes in progs.items()}
        for comp, progs in mc.PROGRAMS.items()
    }
    sym_missing = {f"{c}.{p}": [g for g in gs
                                if not g.startswith("ENSG") and g not in sym2id]
                    for c, ps in mc.PROGRAMS.items() for p, gs in ps.items()}
    n_lost = sum(len(v) for v in sym_missing.values())
    if n_lost:
        warnings.warn(
            f"{n_lost} marker symbol(s) have no ensembl id and were "
            "dropped before scoring; see the 'missing_symbol' column "
            "of the coverage table.")
    mc.PROGRAMS_ENSG = programs        # keep it, but do not clobber

# {"malignant": "Ductal 2"}
if compartment_map is None:
    compartment_map = {"malignant": mc.mal_cell_name}

out, cov = {}, []
for key, cell_type in compartment_map.items():
    if key not in programs:
        warnings.warn(f"no PROGRAMS entry for '{key}'; skipping")
        continue

    M = mc.compartment_matrix(cell_type, samples=samples)

    Zs = (M - M.mean()) / M.std().replace(0, np.nan)

    for prog, genes in programs[key].items():
        found = [g for g in genes if g in Zs.columns]

        cov.append({"compartment": key, "cell_type": cell_type,
                    "program": prog, "n_found": len(found),
                    "n_total": len(genes),
                    "missing": [g for g in genes if g not in Zs.columns],
                    "missing_symbol": sym_missing.get(f"{key}.{prog}", [])})
        if len(found) < min_genes:
            warnings.warn(
                f"{key}/{prog}: only {len(found)}/{len(genes)} markers "
                f"present; skipping (min_genes={min_genes})")
            continue
        out[f"{key}.{prog}"] = Zs[found].mean(axis=1)

scores = pd.DataFrame(out)

# A program score that tracks its own compartment's abundance is an
# abundance readout, not a phenotype.
for rec in cov:
    col = f"{rec['compartment']}.{rec['program']}"
    if col not in scores:
        continue
    th = mc.df_theta[rec["cell_type"]].reindex(scores.index)
    ok = scores[col].notna() & th.notna()
    rec["r_with_theta"] = (round(float(np.corrcoef(
        scores.loc[ok, col], th[ok])[0, 1]), 3) if ok.sum() > 5 else np.nan)
    
# Signed axes are more stable than either pole alone. Name them with
# their OWN compartment prefix so couple_compartments() still treats
# them as belonging to it -- an "axis." prefix would make every axis
# look like a separate compartment and produce trivial self-correlations
# against its own poles.
if "malignant.basal" in scores and "malignant.classical" in scores:
    scores["malignant.axis_basal_minus_classical"] = (
        scores["malignant.basal"] - scores["malignant.classical"])
if "fibroblast.myCAF" in scores and "fibroblast.iCAF" in scores:
    scores["fibroblast.axis_myCAF_minus_iCAF"] = (
        scores["fibroblast.myCAF"] - scores["fibroblast.iCAF"])
if "immune.cytotoxic" in scores and "immune.exhaustion" in scores:
    scores["immune.axis_cytotoxic_minus_exhaustion"] = (
        scores["immune.cytotoxic"] - scores["immune.exhaustion"])

### discretize_axes

### factorial_state_de